In [2]:
from pymatgen.core import Lattice, Structure
import pandas as pd
import numpy as np
import plotly as pt
import seaborn as sns
#!pip install pymatgen
#!pip install mp_api
import requests
import json
import matplotlib.pyplot as plt

In [77]:
root_folder = ""
data_folder = "Data/" #"/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/"
cif_folder = "Data/CIF/"
input_dataset_path = "checkpoint_cif_embeddings.xlsx"
extend_mattergen_dataset = True

In [78]:
df = pd.read_excel(input_dataset_path)

In [79]:
df

,Unnamed: 0,Perovskite,Hill formula,Interlayer space composition,Class,"Bandgap, eV",Materials Project ID,COD_ID,Springer_ID,Z,...,Valence Electrons Density_manual,Oxygen_count,Oxygen_concentration_manual,Oxygen_concentration_MP,Oxygen_concentration_COD,Oxygen_concentration_Springer,MP_packing_fraction,COD_packing_fraction,Springer_packing_fraction,Manual_packing_fraction
0,0,K4Nb6O17,K4 Nb6 O17,NaN,K4Nb6O17,3.50,mp-560692,1001842,-1,4.0,...,0.077010,17.0,0.038505,0.038482,0.040481,NaN,0.482644,0.507705,NaN,0.482925
1,1,KLaNb2O7,K1 La1 Nb2 O7,NaN,HLaNb2O7,3.20,mp-1223501,1545643,-1,8.0,...,0.086868,7.0,0.043434,0.040335,0.021337,NaN,0.484504,0.256300,NaN,0.521726
2,2,RbLaNb2O7,Rb1 La1 Nb2 O7,NaN,HLaNb2O7,3.35,mp-553965,-1,-1,1.0,...,0.084409,7.0,0.042204,0.040114,NaN,NaN,0.507344,NaN,NaN,0.533788
3,3,CsLaNb2O7,Cs1 La1 Nb2 O7,NaN,HLaNb2O7,3.30,mp-553248,2004917,-1,1.0,...,0.082082,7.0,0.041041,0.038958,0.041070,NaN,0.524329,0.552752,NaN,0.552364
4,4,KCa2Nb3O10,K1 Ca2 Nb3 O10,NaN,KCa2Nb3O10,3.35,mp-557195,1521061,-1,8.0,...,0.090945,10.0,0.045472,0.043255,0.045288,NaN,0.505552,0.529316,NaN,0.531466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,1084,TiO2,Ti1 O2,NaN,oxide,3.20,mp-1245098,1010942,-1,NaN,...,NaN,2.0,NaN,0.050730,0.061367,NaN,0.469004,0.567346,NaN,NaN
732,1085,TiO2,Ti1 O2,NaN,oxide,3.20,mp-1245098,1010942,-1,NaN,...,NaN,2.0,NaN,0.050730,0.061367,NaN,0.469004,0.567346,NaN,NaN
733,1086,TiO2,Ti1 O2,NaN,oxide,3.20,mp-1245098,1010942,-1,NaN,...,NaN,2.0,NaN,0.050730,0.061367,NaN,0.469004,0.567346,NaN,NaN
734,1087,TiO2,Ti1 O2,NaN,oxide,3.20,mp-1245098,1010942,-1,NaN,...,NaN,2.0,NaN,0.050730,0.061367,NaN,0.469004,0.567346,NaN,NaN


# Clean my dataset from non-MP entries

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 84 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Unnamed: 0                          736 non-null    int64  
 1   Perovskite                          736 non-null    object 
 2   Hill formula                        736 non-null    object 
 3   Interlayer space composition        4 non-null      object 
 4   Class                               606 non-null    object 
 5   Bandgap, eV                         715 non-null    float64
 6   Materials Project ID                736 non-null    object 
 7   COD_ID                              736 non-null    int64  
 8   Springer_ID                         736 non-null    object 
 9   Z                                   469 non-null    float64
 10  a, A                                484 non-null    float64
 11  b, A                                483 non-n

In [81]:
#df_nan = df[df["MP_CIF_modifier"].isna()]

In [82]:
import os
from pymatgen.io.cif import CifWriter
from mp_api.client import MPRester
API_KEY = "GFsoU5OV3dEngGT860TOtWcn35bE4y6l"

In [83]:
def get_cif_string_from_id(MP_ID):
  
  file_path=cif_folder + str(MP_ID)+".cif"
  #print("Path: ",file_path)
  if os.path.exists(file_path):
    try:
      structure = Structure.from_file(file_path)
    except:
      print('ERROR: Invalid structure for ',MP_ID)
      return None
  else:
    return None

  if(structure == None):
    return None
  writer = CifWriter(structure)
  cif_string = str(writer)
  #print("CIF string: ",cif_string)
  return cif_string

In [84]:
def get_CIF_string_from_IDs(MP_ID, COD_ID, Springer_ID):
    #print("func")
    MP_CIF = get_cif_string_from_id(MP_ID)
    COD_CIF = get_cif_string_from_id(COD_ID)
    Springer_CIF = get_cif_string_from_id(Springer_ID)
    if MP_CIF:
        return MP_CIF
    if COD_CIF:
        return COD_CIF
    if Springer_CIF:
        return Springer_CIF
    return ""

In [85]:
get_CIF_string_from_IDs("mp-31760","","")

"# generated using pymatgen\ndata_Sr2TaFeO6\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   5.66717500\n_cell_length_b   9.81577371\n_cell_length_c   9.81532669\n_cell_angle_alpha   70.70136550\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   Sr2TaFeO6\n_chemical_formula_sum   'Sr8 Ta4 Fe4 O24'\n_cell_volume   515.32350959\n_cell_formula_units_Z   4\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Sr  Sr0  1  0.48739800  0.12345700  0.62621200  1.0\n  Sr  Sr1  1  0.48739800  0.62345700  0.12621200  1.0\n  Sr  Sr2  1  0.98739800  0.87654300  0.87378800  1.0\n  Sr  Sr3  1  0.98739800  0.37654300  0.37378800  1.0\n  Sr  Sr4  1  0.01260200  0.12345700  0.12621200  1.0\n  Sr  Sr5  1  0.01260200  0

In [86]:
df['cif'] = df.apply(lambda row: get_CIF_string_from_IDs(row['MP_CIF_modified'], row['COD_CIF_modified'], row['Springer_CIF_modified']), axis=1)


C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['K1', 'K1', 'K1', 'K1', 'K2', 'K2', 'K2', 'K2', 'K3', 'K3', 'K3', 'K3', 'K4', 'K4', 'K4', 'K4', 'Nb1', 'Nb1', 'Nb1', 'Nb1', 'Nb2', 'Nb2', 'Nb2', 'Nb2', 'Nb3', 'Nb3', 'Nb3', 'Nb3', 'Nb4', 'Nb4', 'Nb4', 'Nb4', 'Nb5', 'Nb5', 'Nb5', 'Nb5', 'Nb6', 'Nb6', 'Nb6', 'Nb6', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2', 'O3', 'O3', 'O3', 'O3', 'O4', 'O4', 'O4', 'O4', 'O5', 'O5', 'O5', 'O5', 'O6', 'O6', 'O6', 'O6', 'O7', 'O7', 'O7', 'O7', 'O8', 'O8', 'O8', 'O8', 'O9', 'O9', 'O9', 'O9', 'O10', 'O10', 'O10', 'O10', 'O11', 'O11', 'O11', 'O11', 'O12', 'O12', 'O12', 'O12', 'O13', 'O13', 'O13', 'O13', 'O14', 'O14', 'O14', 'O14', 'O15', 'O15', 'O15', 'O15', 'O16', 'O16', 'O16', 'O16', 'O17', 'O17', 'O17', 'O17']`.
  writer = CifWriter(structure)
e:\Programming practice\P

ERROR: Invalid structure for  sd_1614775
ERROR: Invalid structure for  sd_1614775
ERROR: Invalid structure for  sd_1614775


C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Rb1', 'Pr1', 'Ta1', 'Ta1', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O3']`.
  writer = CifWriter(structure)
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ta1', 'Ta1', 'Ta1', 'Ta1', 'Ta1', 'Ta1', 'Ta1', 'Ta1', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O5', 'O5', 'O5', 'O5', 'O5', 'O5', 'O5', 'O5', 'O4', 'O4', 'O4', 'O4', 'O4', 

ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412


C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ti0', 'Ti0', 'Ti1', 'Ti1', 'Ti2', 'Ti2', 'Bi3', 'Bi4', 'Bi5', 'Bi6', 'O7', 'O8', 'O9', 'O10', 'O11', 'O12', 'O13', 'O14', 'O15', 'O16', 'O17', 'O18']`.
  writer = CifWriter(structure)
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti2', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Bi1', 'Bi1', 'Bi1', 'Bi1', 'Bi1', 'Bi1', 'Bi1', 'Bi1', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'Bi2', 'O4', 'O4', 'O4', 'O4', 'O4', 'O4', 'O4', 'O4', 'O2', 'O2', '

...

ERROR: Invalid structure for  sd_1241787
ERROR: Invalid structure for  sd_1210175
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1241787
ERROR: Invalid structure for  sd_1210175
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1241787
ERROR: Invalid structure for  sd_1210175
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1241787
ERROR: Invalid structure for  sd_1210175
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1241787
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1925412
ERROR: Invalid structure for  sd_1241787


C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La2', 'La3', 'Ti4', 'Ti5', 'Ti6', 'Na0', 'Na0', 'Na1', 'Na1', 'O7', 'O8', 'O9', 'O10', 'O11', 'O12', 'O13', 'O14', 'O15', 'O16']`.
  writer = CifWriter(structure)
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La2', 'La3', 'Ti4', 'Ti4', 'Ti5', 'Ti5', 'Ti6', 'Ti6', 'Na0', 'Na0', 'Na1', 'Na1', 'O7', 'O8', 'O9', 'O10', 'O11', 'O12', 'O13', 'O14', 'O15', 'O16']`.
  writer = CifWriter(structure)
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__da

ERROR: Invalid structure for  sd_1210175
ERROR: Invalid structure for  sd_1210175


e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
No structure parsed for section 1 in CIF.
'_atom_site_label'
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
No _symmetry_equiv_pos_as_xyz type key found. Defaulting to P1.
  struct = parser.parse_structures(primitive=primitive)[0]
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Cd1', 'Cd1', 'S1', 'S1']`.
  writer = CifWriter(structure)
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2656037478.py:16: UserWarning: S

ERROR: Invalid structure for  sd_1925412


In [88]:
df.to_csv("dataset_cif_strings_generated.csv")

In [89]:
df= df[df["Promoter"] == "No promoter"]
df

,Unnamed: 0,Perovskite,Hill formula,Interlayer space composition,Class,"Bandgap, eV",Materials Project ID,COD_ID,Springer_ID,Z,...,Oxygen_count,Oxygen_concentration_manual,Oxygen_concentration_MP,Oxygen_concentration_COD,Oxygen_concentration_Springer,MP_packing_fraction,COD_packing_fraction,Springer_packing_fraction,Manual_packing_fraction,cif
0,0,K4Nb6O17,K4 Nb6 O17,NaN,K4Nb6O17,3.50,mp-560692,1001842,-1,4.0,...,17.0,0.038505,0.038482,0.040481,NaN,0.482644,0.507705,NaN,0.482925,# generated using pymatgen\ndata_K4Nb6O17\n_sy...
1,1,KLaNb2O7,K1 La1 Nb2 O7,NaN,HLaNb2O7,3.20,mp-1223501,1545643,-1,8.0,...,7.0,0.043434,0.040335,0.021337,NaN,0.484504,0.256300,NaN,0.521726,# generated using pymatgen\ndata_KLaNb2O7\n_sy...
2,2,RbLaNb2O7,Rb1 La1 Nb2 O7,NaN,HLaNb2O7,3.35,mp-553965,-1,-1,1.0,...,7.0,0.042204,0.040114,NaN,NaN,0.507344,NaN,NaN,0.533788,# generated using pymatgen\ndata_RbLaNb2O7\n_s...
3,3,CsLaNb2O7,Cs1 La1 Nb2 O7,NaN,HLaNb2O7,3.30,mp-553248,2004917,-1,1.0,...,7.0,0.041041,0.038958,0.041070,NaN,0.524329,0.552752,NaN,0.552364,# generated using pymatgen\ndata_CsLaNb2O7\n_s...
4,4,KCa2Nb3O10,K1 Ca2 Nb3 O10,NaN,KCa2Nb3O10,3.35,mp-557195,1521061,-1,8.0,...,10.0,0.045472,0.043255,0.045288,NaN,0.505552,0.529316,NaN,0.531466,# generated using pymatgen\ndata_KCa2Nb3O10\n_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,1067,LiNdNb2O7,Li1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.58,-1,-1,sd_1150217,1.0,...,7.0,0.045922,NaN,NaN,0.040702,NaN,NaN,0.416415,0.469821,# generated using pymatgen\ndata_LiNdNb2O7\n_s...
715,1068,NaNdNb2O7,Na1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.57,-1,-1,sd_1150217,1.0,...,7.0,0.045041,NaN,NaN,0.040702,NaN,NaN,0.436677,0.483225,# generated using pymatgen\ndata_NaNdNb2O7\n_s...
716,1069,RbNdNb2O7,Rb1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.55,-1,-1,sd_1150217,1.0,...,7.0,0.042939,NaN,NaN,0.040702,NaN,NaN,0.510072,0.538106,# generated using pymatgen\ndata_RbNdNb2O7\n_s...
717,1070,CsNdNb2O7,Cs1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.62,-1,-1,sd_1150217,1.0,...,7.0,0.042320,NaN,NaN,0.040702,NaN,NaN,0.543085,0.564668,# generated using pymatgen\ndata_CsNdNb2O7\n_s...


In [90]:
#df = df[df["Materials Project ID"].str.contains(r"^mp-",na=False)]

In [91]:
#df.to_excel("checkpoint_only_MP.xlsx", index=False)

# Clean my dataset from duplicates

In [92]:
df

,Unnamed: 0,Perovskite,Hill formula,Interlayer space composition,Class,"Bandgap, eV",Materials Project ID,COD_ID,Springer_ID,Z,...,Oxygen_count,Oxygen_concentration_manual,Oxygen_concentration_MP,Oxygen_concentration_COD,Oxygen_concentration_Springer,MP_packing_fraction,COD_packing_fraction,Springer_packing_fraction,Manual_packing_fraction,cif
0,0,K4Nb6O17,K4 Nb6 O17,NaN,K4Nb6O17,3.50,mp-560692,1001842,-1,4.0,...,17.0,0.038505,0.038482,0.040481,NaN,0.482644,0.507705,NaN,0.482925,# generated using pymatgen\ndata_K4Nb6O17\n_sy...
1,1,KLaNb2O7,K1 La1 Nb2 O7,NaN,HLaNb2O7,3.20,mp-1223501,1545643,-1,8.0,...,7.0,0.043434,0.040335,0.021337,NaN,0.484504,0.256300,NaN,0.521726,# generated using pymatgen\ndata_KLaNb2O7\n_sy...
2,2,RbLaNb2O7,Rb1 La1 Nb2 O7,NaN,HLaNb2O7,3.35,mp-553965,-1,-1,1.0,...,7.0,0.042204,0.040114,NaN,NaN,0.507344,NaN,NaN,0.533788,# generated using pymatgen\ndata_RbLaNb2O7\n_s...
3,3,CsLaNb2O7,Cs1 La1 Nb2 O7,NaN,HLaNb2O7,3.30,mp-553248,2004917,-1,1.0,...,7.0,0.041041,0.038958,0.041070,NaN,0.524329,0.552752,NaN,0.552364,# generated using pymatgen\ndata_CsLaNb2O7\n_s...
4,4,KCa2Nb3O10,K1 Ca2 Nb3 O10,NaN,KCa2Nb3O10,3.35,mp-557195,1521061,-1,8.0,...,10.0,0.045472,0.043255,0.045288,NaN,0.505552,0.529316,NaN,0.531466,# generated using pymatgen\ndata_KCa2Nb3O10\n_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,1067,LiNdNb2O7,Li1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.58,-1,-1,sd_1150217,1.0,...,7.0,0.045922,NaN,NaN,0.040702,NaN,NaN,0.416415,0.469821,# generated using pymatgen\ndata_LiNdNb2O7\n_s...
715,1068,NaNdNb2O7,Na1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.57,-1,-1,sd_1150217,1.0,...,7.0,0.045041,NaN,NaN,0.040702,NaN,NaN,0.436677,0.483225,# generated using pymatgen\ndata_NaNdNb2O7\n_s...
716,1069,RbNdNb2O7,Rb1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.55,-1,-1,sd_1150217,1.0,...,7.0,0.042939,NaN,NaN,0.040702,NaN,NaN,0.510072,0.538106,# generated using pymatgen\ndata_RbNdNb2O7\n_s...
717,1070,CsNdNb2O7,Cs1 Nd1 Nb2 O7,NaN,HLaNb2O7,3.62,-1,-1,sd_1150217,1.0,...,7.0,0.042320,NaN,NaN,0.040702,NaN,NaN,0.543085,0.564668,# generated using pymatgen\ndata_CsNdNb2O7\n_s...


In [93]:
df['Alcohol_nonzero'] = df['Alcohol, %'].replace(0, np.nan)

C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2565595144.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Alcohol_nonzero'] = df['Alcohol, %'].replace(0, np.nan)


In [94]:
df = df[df['Alcohol_nonzero'].notna()]

In [95]:
idx = df.groupby('Materials Project ID')['Alcohol_nonzero'].idxmin()
idx

Materials Project ID
-1                   525
HCa2Ta3O10_BuNH2     601
HCa2Ta3O10_MeNH2     600
HCa2Ta3O10_OcNH2     602
HLaTiO4_BuNH2        448
HLaTiO4_BuOH         453
HLaTiO4_EtNH2        446
HLaTiO4_EtOH         451
HLaTiO4_HxNH2        449
HLaTiO4_HxOH         454
HLaTiO4_MeNH2        445
HLaTiO4_MeOH         450
HLaTiO4_PrNH2        447
HLaTiO4_PrOH         452
K2La2Ti3O10_BuNH2    535
K2La2Ti3O10_BuOH     541
K2La2Ti3O10_EtNH2    533
K2La2Ti3O10_EtOH     539
K2La2Ti3O10_HxNH2    536
K2La2Ti3O10_HxOH     542
K2La2Ti3O10_MeNH2    532
K2La2Ti3O10_MeOH     538
K2La2Ti3O10_OcNH2    629
K2La2Ti3O10_PrNH2    534
K2La2Ti3O10_PrOH     540
mp-10347             488
mp-1104930           694
mp-1179025           289
mp-1205881           275
mp-1222828           229
mp-1223501             1
mp-1223520           213
mp-1223975           430
mp-1245098           703
mp-20396               5
mp-23611              52
mp-23614              54
mp-27998              34
mp-4423              696
mp-5

In [96]:
df = df.loc[idx].copy()
df

,Unnamed: 0,Perovskite,Hill formula,Interlayer space composition,Class,"Bandgap, eV",Materials Project ID,COD_ID,Springer_ID,Z,...,Oxygen_concentration_manual,Oxygen_concentration_MP,Oxygen_concentration_COD,Oxygen_concentration_Springer,MP_packing_fraction,COD_packing_fraction,Springer_packing_fraction,Manual_packing_fraction,cif,Alcohol_nonzero
525,777,HNdTa2O7,H1 Nd1 Ta2 O7,NaN,HLaNb2O7,4.360000,-1,-1,sd_1955780,1.0,...,0.044141,NaN,NaN,0.046391,NaN,NaN,0.454380,0.432338,# generated using pymatgen\ndata_NdTa2HO7\n_sy...,0.10
601,903,HCa2Nb3O10*2C4H9NH2,H23 Ca2 Nb3 O10 C8 N2,NaN,KCa2Nb3O10,3.580000,HCa2Ta3O10_BuNH2,-1,-1,1.0,...,0.026527,0.021705,NaN,NaN,0.265534,NaN,NaN,0.324529,# generated using pymatgen\ndata_Ca2Nb3H24C8(N...,1.00
600,902,HCa2Nb3O10*2CH3NH2,H11 Ca2 Nb3 O10 C2 N2,NaN,KCa2Nb3O10,3.520000,HCa2Ta3O10_MeNH2,-1,-1,1.0,...,0.041423,0.030833,NaN,NaN,0.375123,NaN,NaN,0.503960,# generated using pymatgen\ndata_Ca2Nb3H12C2(N...,1.00
602,904,HCa2Nb3O10*2C8H17NH2,H39 Ca2 Nb3 O10 C16 N2,NaN,KCa2Nb3O10,3.520000,HCa2Ta3O10_OcNH2,-1,-1,1.0,...,0.019341,0.018411,NaN,NaN,0.226913,NaN,NaN,0.238370,# generated using pymatgen\ndata_Ca2Nb3H40C16(...,1.00
448,692,HLaTiO4*C4H9NH2,H12 La1 Ti1 O4 C4 N1,NaN,HLaTiO4,3.610000,HLaTiO4_BuNH2,-1,-1,4.0,...,0.023676,0.022345,NaN,NaN,0.290927,NaN,NaN,0.308260,# generated using pymatgen\ndata_LaTiH12C4NO4\...,1.00
453,698,HLaTiO4*C4H9OH,H11 La1 Ti1 O5 C4,NaN,HLaTiO4,3.450000,HLaTiO4_BuOH,-1,-1,4.0,...,0.031524,0.027931,NaN,NaN,0.283917,NaN,NaN,0.320439,# generated using pymatgen\ndata_LaTiH11C4O5\n...,1.00
446,690,HLaTiO4*C2H5NH2,H8 La1 Ti1 O4 C2 N1,NaN,HLaTiO4,3.480000,HLaTiO4_EtNH2,-1,-1,4.0,...,0.030503,0.028635,NaN,NaN,0.371203,NaN,NaN,0.395420,# generated using pymatgen\ndata_LaTiH8C2NO4\n...,1.00
451,696,HLaTiO4*C2H5OH,H7 La1 Ti1 O5 C2,NaN,HLaTiO4,3.470000,HLaTiO4_EtOH,-1,-1,4.0,...,0.039488,0.035794,NaN,NaN,0.362219,NaN,NaN,0.399597,# generated using pymatgen\ndata_LaTiH7C2O5\n_...,1.00
449,693,HLaTiO4*C6H13NH2,H16 La1 Ti1 O4 C6 N1,NaN,HLaTiO4,3.470000,HLaTiO4_HxNH2,-1,-1,4.0,...,0.020614,0.019455,NaN,NaN,0.254403,NaN,NaN,0.269561,# generated using pymatgen\ndata_LaTiH16C6NO4\...,1.00
454,699,HLaTiO4*C6H13OH,H15 La1 Ti1 O5 C6,NaN,HLaTiO4,3.460000,HLaTiO4_HxOH,-1,-1,4.0,...,0.025809,0.024319,NaN,NaN,0.248300,NaN,NaN,0.263514,# generated using pymatgen\ndata_LaTiH15C6O5\n...,1.00


In [97]:
idx

Materials Project ID
-1                   525
HCa2Ta3O10_BuNH2     601
HCa2Ta3O10_MeNH2     600
HCa2Ta3O10_OcNH2     602
HLaTiO4_BuNH2        448
HLaTiO4_BuOH         453
HLaTiO4_EtNH2        446
HLaTiO4_EtOH         451
HLaTiO4_HxNH2        449
HLaTiO4_HxOH         454
HLaTiO4_MeNH2        445
HLaTiO4_MeOH         450
HLaTiO4_PrNH2        447
HLaTiO4_PrOH         452
K2La2Ti3O10_BuNH2    535
K2La2Ti3O10_BuOH     541
K2La2Ti3O10_EtNH2    533
K2La2Ti3O10_EtOH     539
K2La2Ti3O10_HxNH2    536
K2La2Ti3O10_HxOH     542
K2La2Ti3O10_MeNH2    532
K2La2Ti3O10_MeOH     538
K2La2Ti3O10_OcNH2    629
K2La2Ti3O10_PrNH2    534
K2La2Ti3O10_PrOH     540
mp-10347             488
mp-1104930           694
mp-1179025           289
mp-1205881           275
mp-1222828           229
mp-1223501             1
mp-1223520           213
mp-1223975           430
mp-1245098           703
mp-20396               5
mp-23611              52
mp-23614              54
mp-27998              34
mp-4423              696
mp-5

In [98]:
df['Rate_standardized'] = df['Rate, umol/(g*h)'] / (df['Alcohol, %']*df['CatW, g/L']*df["Power, W"])

In [99]:
df["Log_rate_standardized"] = np.log(df['Rate_standardized'])

In [100]:
df.to_excel("pre_dataset_for_fine_tuning.xlsx", index=False)

In [101]:
df.shape[0]

57

# Changing mattergen datasets

In [102]:
/0

<>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\1350904145.py:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
  0()


TypeError: 'int' object is not callable

In [103]:
def add_value_to_dataset_MP(MP_ID):
    row = df[df['Materials Project ID'] == MP_ID]
    if row.empty:
        return np.nan
    else:
        return row['Log_rate_standardized'].values[0]

In [104]:
if not extend_mattergen_dataset:
    names = ["train.csv"]
    for name in names:
        df_mattergen_set = pd.read_csv("Data/Mattergen_dataset/init/"+name)
        print(name + ": " + str(df_mattergen_set.shape[0]))
        df_mattergen_set["Log_rate"] = df_mattergen_set["material_id"].apply(add_value_to_dataset)
        #df_mattergen_set = df_mattergen_set[df_mattergen_set["Log_rate"].notna()]
        print(name + ": " + str(df_mattergen_set.shape[0]))
        df_mattergen_set.to_csv("Data/Mattergen_dataset/processed/"+name, index=False)

In [105]:
print(get_cif_string_from_id("mp-6144"))

# generated using pymatgen
data_Na2La2Ti3O10
_symmetry_space_group_name_H-M   'P 1'
_cell_length_a   14.69641334
_cell_length_b   14.69641334
_cell_length_c   14.69641334
_cell_angle_alpha   164.86303529
_cell_angle_beta   164.86303529
_cell_angle_gamma   21.47009583
_symmetry_Int_Tables_number   1
_chemical_formula_structural   Na2La2Ti3O10
_chemical_formula_sum   'Na2 La2 Ti3 O10'
_cell_volume   216.40687731
_cell_formula_units_Z   1
loop_
 _symmetry_equiv_pos_site_id
 _symmetry_equiv_pos_as_xyz
  1  'x, y, z'
loop_
 _atom_site_type_symbol
 _atom_site_label
 _atom_site_symmetry_multiplicity
 _atom_site_fract_x
 _atom_site_fract_y
 _atom_site_fract_z
 _atom_site_occupancy
  Na  Na0  1  0.71177800  0.71177800  0.00000000  1.0
  Na  Na1  1  0.28822200  0.28822200  0.00000000  1.0
  La  La2  1  0.57527700  0.57527700  0.00000000  1.0
  La  La3  1  0.42472300  0.42472300  0.00000000  1.0
  Ti  Ti4  1  0.85219600  0.85219600  0.00000000  1.0
  Ti  Ti5  1  0.14780400  0.14780400  0.00000000

In [106]:
def get_material_data_from_id(MP_ID):
    output = {}
    with MPRester(API_KEY) as mpr:
        data = mpr.summary.get_data_by_id(MP_ID)
        #print(f"Formula: {data.formula_pretty}")
        #print(f"Formation Energy: {data.formation_energy_per_atom} eV/atom")
        #print(f"Band Gap: {data.band_gap}")
        #print(f"Formula: {data.formula_pretty}")
        #try:
        #    #print(f"e_above_hull: {data.e_above_hull}")
        #except:
            #print("e_above_hull data not available") 
        elements = data.elements
        element_labels = [el.symbol for el in elements]
        #print(f"Elements: {element_labels}")
        #print(f"Space Group: {data.symmetry.number}")
        #************************
        #************************
        output["pretty_formula"] = data.formula_pretty
        output["formation_energy_per_atom"] = data.formation_energy_per_atom
        output["dft_band_gap"] = data.band_gap
        output["e_above_hull"] = data.e_above_hull if hasattr(data, 'e_above_hull') else None
        output["elements"] = element_labels
        output["spacegroup_number"] = data.symmetry.number
    return output

In [107]:
get_material_data_from_id("mp-6144")

C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2937985821.py:4: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  data = mpr.summary.get_data_by_id(MP_ID)
C:\Users\Nikita\AppData\Local\Temp\ipykernel_21016\2937985821.py:4: DeprecationWarning: get_data_by_id is deprecated and will be removed soon. Please use the search method instead.
  data = mpr.summary.get_data_by_id(MP_ID)
Retrieving SummaryDoc documents: 100%|██████████| 1/1 [00:00<00:00, 10082.46it/s]


{'pretty_formula': 'Na2La2Ti3O10',
 'formation_energy_per_atom': -3.412753374868154,
 'dft_band_gap': 1.7671000000000001,
 'e_above_hull': None,
 'elements': ['La', 'Na', 'O', 'Ti'],
 'spacegroup_number': 139}

In [ ]:
added = 0
replaced = 0

In [ ]:
def add_extend_value_to_mattergen_dataset(mattergen_dataset, MP_ID, Log_rate):
    global added, replaced
    #print(my_row)
    #MP_ID = my_row['Materials Project ID']
    #Log_rate =  my_row['Log_rate']
    #print("MP_ID: ",MP_ID)
    #print("Log_rate: ",Log_rate)
    is_present = MP_ID in mattergen_dataset['material_id'].values
    #print("MP_ID is in dataset: ",is_present)
    if(is_present):
        with open("dataset_extension_log.txt", "a") as file:
            file.write(f"MP_ID: {MP_ID}, Action: replaced\n")
        mattergen_dataset.loc[mattergen_dataset['material_id'] == MP_ID, 'Log_rate'] = Log_rate
        mattergen_dataset.loc[mattergen_dataset['material_id'] == MP_ID, 'Action'] = "replaced"
        replaced = replaced + 1
    else:
        with open("dataset_extension_log.txt", "a") as file:
            file.write("MP_ID: ", MP_ID , ", Action: added\n")
        new_row = {'material_id': MP_ID, 'Log_rate': Log_rate, 'Action':"added"}
        additional_info = get_material_data_from_id(MP_ID)
        new_row.update(additional_info)
        #mattergen_dataset.loc[len(mattergen_dataset)] = new_row
        new_row_df = pd.DataFrame([new_row])
        print(mattergen_dataset.shape[0])
        print(new_row_df.shape[0])
        mattergen_dataset = pd.concat([mattergen_dataset, new_row_df], ignore_index=True)
        print(mattergen_dataset.shape[0])
        added = added + 1
    return mattergen_dataset

In [ ]:
if (extend_mattergen_dataset):
    names = ["train.csv"]
    #names = ["val.csv"]
    with open("dataset_extension_log.txt", "w") as file:
            file.write("***********************************************\n")
    for name in names:
        with open("dataset_extension_log.txt", "a") as file:
            file.write("***********************************************\n")
        df_mattergen_set = pd.read_csv("Data/Mattergen_dataset/init/"+name)

        #print(name + ": " + str(df_mattergen_set.shape[0]))

        my_count = 0
        for row in df.to_dict('records'):
            mp_id = row['Materials Project ID']
            log_rate = row['Log_rate']
            df_mattergen_set = add_extend_value_to_mattergen_dataset(df_mattergen_set, mp_id, log_rate)
            my_count += 1
        #print(name + ": " + str(df_mattergen_set.shape[0]))
        df_mattergen_set = df_mattergen_set.dropna(subset=['Log_rate'])
        #print(name + ": " + str(df_mattergen_set.shape[0]))
        df_mattergen_set["cif"]=df_mattergen_set["material_id"].apply(get_cif_string_from_id)
        df_mattergen_set.to_csv("Data/Mattergen_dataset/processed/"+name, index=False)
        print(f"File: {name}, Added: {added}, Replaced: {replaced}, My count: {my_count}")
        added = 0
        replaced = 0

    #for row in df.itertuples():
    #    MP_ID = getattr(row, 'Materials_Project_ID') 
    #    Log_rate = getattr(row, 'Log_rate') 
    #    print("MP_ID: ", MP_ID)
    #    print("Log_rate: ", Log_rate)
    #    add_extend_value_to_mattergen_dataset(df_mattergen_set,MP_ID,Log_rate)

TypeError: TextIOWrapper.write() takes exactly one argument (3 given)

In [ ]:
name = "train"
df_ = pd.read_csv(f"Data/Mattergen_dataset/processed/{name}.csv")
df_

,Unnamed: 0,material_id,formation_energy_per_atom,dft_band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup_number,azure_bulk_modulus,larsen_score_2d,Si_100_mismatch,azure_band_gap,dft_bulk_modulus,dft_poisson_ratio,dft_mag_density,Log_rate
0,0.0,mp-6144,-3.415012,2.0830,Na2La2Ti3O10,0.012505,"['La', 'Na', 'O', 'Ti']",# generated using pymatgen\ndata_Na2La2Ti3O10\...,139.0,119.400684,7.515893e-03,0.003792,2.113394,NaN,NaN,1.280000e-05,4.605170
1,1.0,mp-1222828,-2.894919,1.2000,LaNb2AgO7,0.064722,"['Ag', 'La', 'Nb', 'O']",# generated using pymatgen\ndata_LaNb2AgO7\n_s...,119.0,140.181002,7.770000e-16,0.004277,1.123539,NaN,NaN,6.380000e-06,5.337538
2,2.0,mp-541600,-3.383709,2.3257,RbLaTa2O7,0.021469,"['La', 'O', 'Rb', 'Ta']",# generated using pymatgen\ndata_RbLaTa2O7\n_s...,123.0,97.102628,3.430501e-01,0.008662,2.345044,NaN,NaN,4.330000e-07,2.740840
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_ = df_.iloc[:3, :]
df_

,Unnamed: 0,material_id,formation_energy_per_atom,dft_band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup_number,azure_bulk_modulus,larsen_score_2d,Si_100_mismatch,azure_band_gap,dft_bulk_modulus,dft_poisson_ratio,dft_mag_density,Log_rate
0,0.0,mp-6144,-3.415012,2.0830,Na2La2Ti3O10,0.012505,"['La', 'Na', 'O', 'Ti']",# generated using pymatgen\ndata_Na2La2Ti3O10\...,139.0,119.400684,7.515893e-03,0.003792,2.113394,NaN,NaN,1.280000e-05,4.605170
1,1.0,mp-1222828,-2.894919,1.2000,LaNb2AgO7,0.064722,"['Ag', 'La', 'Nb', 'O']",# generated using pymatgen\ndata_LaNb2AgO7\n_s...,119.0,140.181002,7.770000e-16,0.004277,1.123539,NaN,NaN,6.380000e-06,5.337538
2,2.0,mp-541600,-3.383709,2.3257,RbLaTa2O7,0.021469,"['La', 'O', 'Rb', 'Ta']",# generated using pymatgen\ndata_RbLaTa2O7\n_s...,123.0,97.102628,3.430501e-01,0.008662,2.345044,NaN,NaN,4.330000e-07,2.740840


In [ ]:
df_.to_csv(f"Data/Mattergen_dataset/processed/{name}.csv")